# 02 Variable audit: Haiti

Look at each variable **one by one** before it goes into anything else: fetch it, map it, and read its numbers.
The variables are organised in four groups, and in subgroups inside them:

| Group | Subgroups | Variables |
|---|---|---|
| Climate variables | Rainfall and temperature, Dryness | annual_precipitation, mean_annual_temperature, aridity_index |
| Soil | Fertility, Water | soil_ph, soil_organic_carbon, soil_drainage |
| Terrain and land | Topography, Land cover and vegetation | slope, land_cover, tree_canopy_cover |
| Project risk | Access and energy, Legal and protection | accessibility_travel_time, distance_to_road, electrification_index, protected_area_status |

For every variable you get the dataset it comes from, where it was read (a local copy, the Hugging Face dataset, Earth Engine),
its range, the share of the area with no data, a map and a histogram, and a **CHECK** line when something looks wrong
(for example values outside a plausible range, which catches a missing or doubled scale factor).
Nothing is replaced by made-up data: a variable that cannot be loaded says why.

The last section is a summary table, grouped, that also shows the role each variable plays in CIAT's pinned package.
That role is read from the package, so it follows CIAT's updates.

In [ ]:
# ===== PARAMETERS: edit only this cell =====
AOI_NAME = "Haiti (approximate box)"
AOI_BBOX = (-74.5, 18.0, -71.6, 20.1)       # minx, miny, maxx, maxy in EPSG:4326
RESOLUTION_M = 5000                         # 5 km: quick. Finer grids make the first Earth Engine extraction much slower

GROUPS = ["Climate variables", "Soil", "Terrain and land", "Project risk"]   # which groups to run
SKIP = []                                   # variable ids to leave out, for example ["land_cover"]
INCLUDE_SLOW = True                         # False leaves out the variables that read 30 m or 10 m data in Earth Engine

DATA_SOURCE = "real"                        # "real", or "synthetic" to test the notebook offline (never real results)
DATA_ORDER = ("local", "hf")                # where a dataset file is looked for, in this order
LOCAL_DATA_DIR = ""                         # optional: a folder with a copy of the dataset (same layout: raw/<folder>/<file>)
HF_REPO = "dagudelo30/nbs-ruralscan-data"
HF_MODE = "stream"                          # "stream": read only the window over HTTP | "download": fetch whole files once

EE_PROJECT = ""                             # leave empty: the project id is read from EE_PROJECT in .env
USE_CACHE = True                            # Earth Engine layers are extracted once and kept in cache/layers/
CACHE_DIR = "cache/layers"
GEE_WORKERS = 4                             # Earth Engine requests in parallel
GEE_TILE_INPUT_PIXELS = 2_500_000           # lower it if you see "User memory limit exceeded"
VIIRS_YEAR = 2023                           # year of the night-time lights composite
POLICY_ID = "planted_terrain_soil_v1"       # the policy used to label each variable's role

SAVE_FIGURES = True                         # saves each figure in runs/audit-.../

## 0. Environment

Finds the repository root, loads `nbs_runtime` (from `src/` if it is not installed), and checks that the packages this run needs
are available **in this Python**. What is needed depends on the variables you selected.

In [ ]:
import importlib.util
import sys
from pathlib import Path


def find_repo_root():
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / "src" / "nbs_runtime").is_dir() and (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find the nbs-ruralscan-runtime repository above {here}. "
                       "Open this notebook from inside the repository folder.")


REPO_ROOT = find_repo_root()
print("python:", sys.executable)
print("repository:", REPO_ROOT)
if importlib.util.find_spec("nbs_runtime") is None:
    sys.path.insert(0, str(REPO_ROOT / "src"))
    print("nbs_runtime is not installed in this Python: loading it from", REPO_ROOT / "src")

from nbs_runtime import variables as V
from nbs_runtime.selfcheck import assert_in_sync

assert_in_sync()                  # fails with the list of out-of-date files if an update was applied only in part

selected = V.select(GROUPS, SKIP, INCLUDE_SLOW)
sources = {v.source for v in selected}
needed = {"numpy": "numpy", "matplotlib": "matplotlib"}
if DATA_SOURCE == "real":
    if sources & {V.HF_RASTER, V.HWSD}:
        needed["rasterio"] = "rasterio"
    if V.OSM_ROADS in sources:
        needed |= {"rasterio": "rasterio", "geopandas": "geopandas", "shapely": "shapely", "pyproj": "pyproj"}
    if HF_MODE == "download" or sources & {V.HWSD, V.OSM_ROADS}:
        needed["huggingface_hub"] = "huggingface_hub"
    if sources & {V.GEE_CACHED, V.GEE_IMAGE}:
        needed |= {"ee": "earthengine-api", "xee": "xee", "xarray": "xarray"}
missing = [pip_name for mod, pip_name in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    raise ModuleNotFoundError("Missing in this Python: " + ", ".join(sorted(set(missing))) +
                              f"\nInstall with:  {sys.executable} -m pip install " + " ".join(sorted(set(missing))) +
                              "\nthen restart the kernel.")
print(f"environment OK: {len(selected)} variables selected in {len({v.group for v in selected})} group(s)")

## Setup

In [ ]:
from dataclasses import replace

import matplotlib.pyplot as plt

from nbs_runtime.audit import audit_data, describe, summary_rows
from nbs_runtime.contract import load_package
from nbs_runtime.data import gee, hf
from nbs_runtime.data.cache import LayerCache
from nbs_runtime.data.variables_loader import LoadContext, load_variable
from nbs_runtime.grid import Grid
from nbs_runtime.viz import plot_group_overview, plot_variable_audit

grid = Grid.from_bbox(AOI_BBOX, RESOLUTION_M)
OUT_DIR = REPO_ROOT / "runs" / f"audit-{DATA_SOURCE}-{RESOLUTION_M}m"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"grid: {grid.shape[0]} x {grid.shape[1]} cells at {RESOLUTION_M} m | outputs in {OUT_DIR.relative_to(REPO_ROOT)}")

local_roots = [REPO_ROOT / "data"] + ([Path(LOCAL_DATA_DIR)] if LOCAL_DATA_DIR else [])
cache = LayerCache(REPO_ROOT / CACHE_DIR, grid, gee.ASSET_IDS) if (USE_CACHE and DATA_SOURCE == "real") else None
ctx = LoadContext(grid, DATA_ORDER, local_roots, HF_REPO, HF_MODE, cache, GEE_WORKERS, GEE_TILE_INPUT_PIXELS,
                  verbose=True, synthetic=(DATA_SOURCE == "synthetic"))

# the selected variables (the lights year is a parameter)
SPECS = {v.id: (replace(v, detail={**v.detail, "year": VIIRS_YEAR}) if v.id == "electrification_index" else v)
         for v in selected}
pkg = load_package()
roles = V.contract_roles(pkg)
POLICY_VARS = V.policy_variables(pkg, POLICY_ID)
RESULTS = {}

if DATA_SOURCE == "real":
    if {v.source for v in selected} & {V.GEE_CACHED, V.GEE_IMAGE}:
        print("Earth Engine project:", gee.init_ee(EE_PROJECT))
    if {v.source for v in selected} & {V.HF_RASTER, V.HWSD, V.OSM_ROADS} and "hf" in DATA_ORDER:
        print("Hugging Face dataset:", hf.check_access(HF_REPO))


def audit(var_id):
    """Load one variable, print its numbers and draw it. The same call is used for every variable below."""
    if var_id not in SPECS:
        print(f"{var_id}: not selected (see GROUPS, SKIP and INCLUDE_SLOW)")
        return None
    a = audit_data(load_variable(SPECS[var_id], ctx))
    RESULTS[var_id] = a
    print(describe(a))
    plot_variable_audit(a, grid, OUT_DIR / f"{var_id}.png" if SAVE_FIGURES else None)
    plt.show()
    return a


def group_overview(group):
    """All the variables of one group side by side."""
    got = [RESULTS[v.id] for v in V.CATALOG if v.group == group and v.id in RESULTS]
    if not got:
        print(f"{group}: nothing was audited")
        return
    plot_group_overview(got, grid, group, OUT_DIR / f"group_{group.replace(' ', '_').lower()}.png" if SAVE_FIGURES else None)
    plt.show()

# 1. Climate variables

Rainfall, temperature and dryness. They come from global rasters hosted in the Hugging Face dataset, and only the window over the area is read.

## 1.1 Rainfall and temperature

### `annual_precipitation`: WorldClim 2.1, BIO12

Unit: **mm/year**. Native resolution: 1000 m. Read from: Hugging Face dataset.

**What to look for.** Hundreds to a few thousand mm per year. A map with a single flat colour deserves a second look.

*1970-2000 average. CIAT's default binding for this variable points to CHELSA; this is the WorldClim file hosted in the dataset. Confirm which one the policy should use.*

In [ ]:
audit("annual_precipitation")

### `mean_annual_temperature`: WorldClim 2.1, BIO1

Unit: **degC**. Native resolution: 1000 m. Read from: Hugging Face dataset.

**What to look for.** Degrees Celsius (not tenths of a degree). Cooler on the highest ground.

*1970-2000 average. CIAT's default binding points to CHELSA; this is the WorldClim file.*

In [ ]:
audit("mean_annual_temperature")

## 1.2 Dryness

### `aridity_index`: Global Aridity Index v3.1

Unit: **dimensionless**. Native resolution: 1000 m. Read from: Hugging Face dataset.

**What to look for.** Between 0 and about 2. Values in the thousands mean the 0.0001 factor was not applied.

*Stored as integers x 10,000; multiplied by 0.0001 here. Low values are dry, high are humid.*

In [ ]:
audit("aridity_index")

In [ ]:
group_overview("Climate variables")

# 2. Soil

Soil chemistry (pH, organic carbon) from SoilGrids in Earth Engine, and drainage from HWSD v2 in the Hugging Face dataset.

## 2.1 Fertility

### `soil_ph`: SoilGrids 2.0

Unit: **pH units**. Native resolution: 250 m. Read from: Earth Engine (cached).

**What to look for.** About 4 to 9. Values like 40 to 90 mean the conversion from pH x 10 was missed.

*Three depths, converted once (pH x 10 to pH) and weighted by layer thickness.*

In [ ]:
audit("soil_ph")

### `soil_organic_carbon`: SoilGrids 2.0

Unit: **g/kg**. Native resolution: 250 m. Read from: Earth Engine (cached).

**What to look for.** g/kg after conversion. CIAT's function expects g/kg (the earlier prototype converted to percent, a different unit). Look at the real min, max and mean.

*Three depths, converted once (dg/kg x 0.1 to g/kg) and weighted by layer thickness.*

In [ ]:
audit("soil_organic_carbon")

## 2.2 Water

### `soil_drainage`: HWSD v2

Unit: **class 1-7**. Native resolution: 1000 m. Read from: Hugging Face dataset (raster + attribute database).

**What to look for.** Whole classes from 1 to 7, never fractions. A smooth gradient would mean the soil-unit ids were averaged.

*The raster holds soil-unit ids; the drainage class comes from the attribute database, taking the dominant soil of each unit.*

In [ ]:
audit("soil_drainage")

In [ ]:
group_overview("Soil")

# 3. Terrain and land

Slope, land cover and tree canopy from Earth Engine. These read 30 m and 10 m data, so the first extraction is slow; it is cached afterwards.

## 3.1 Topography

### `slope`: SRTM

Unit: **degrees**. Native resolution: 30 m. Read from: Earth Engine (cached).

**What to look for.** Degrees. The mountains stand out. The sea reads 0, and the soil layers are what mark it as no data.

*Derived at the native 30 m scale and only then averaged to the grid.*

In [ ]:
audit("slope")

## 3.2 Land cover and vegetation

### `land_cover`: ESA WorldCover 2021

Unit: **class**. Native resolution: 10 m. Read from: Earth Engine (cached).

**What to look for.** A handful of discrete classes. A smooth gradient would mean a class code was averaged.

*Shows the class with the largest share in each cell. The host gate uses the share of each class, not the dominant one.*

In [ ]:
audit("land_cover")

### `tree_canopy_cover`: Hansen Global Forest Change (treecover2000)

Unit: **percent**. Native resolution: 30 m. Read from: Earth Engine.

**What to look for.** 0 to 100 percent.

*Canopy cover in the year 2000. Newer versions of the asset exist; this is the one in CIAT's registry.*

In [ ]:
audit("tree_canopy_cover")

In [ ]:
group_overview("Terrain and land")

# 4. Project risk

Access, energy and protection: the conditions that decide whether a project can be carried out, rather than whether the land is suitable. CIAT keeps these outside the suitability score.

## 4.1 Access and energy

### `accessibility_travel_time`: Oxford MAP accessibility 2015

Unit: **minutes**. Native resolution: 1000 m. Read from: Earth Engine.

**What to look for.** Minutes. Large values are normal far from cities. Do not expect a 0-1 scale here.

*Raw minutes. Any transformation to a 0-1 scale belongs to the function that uses it.*

In [ ]:
audit("accessibility_travel_time")

### `distance_to_road`: OpenStreetMap roads (Geofabrik)

Unit: **km**. Native resolution: 100 m. Read from: Hugging Face dataset (road shapefile).

**What to look for.** Kilometres, zero on a road. The file also holds the Dominican Republic's roads, so cells near the border see them.

*Straight-line distance to the nearest road of any class, in a local metric projection. The file covers Haiti and the Dominican Republic.*

In [ ]:
audit("distance_to_road")

### `electrification_index`: VIIRS annual night-time lights

Unit: **nW/cm2/sr**. Native resolution: 500 m. Read from: Earth Engine.

**What to look for.** Mostly close to 0, with bright spots in towns and cities.

*Most rural land is near 0 and cities stand out.*

In [ ]:
audit("electrification_index")

## 4.2 Legal and protection

### `protected_area_status`: WDPA (World Database on Protected Areas)

Unit: **share 0-1**. Native resolution: 30 m. Read from: Earth Engine.

**What to look for.** A share between 0 and 1 of each cell inside a protected area. Context only, not a legal rule.

*Context only. CIAT states that a protected-area category alone does not establish a legal exclusion: that needs an assessed rule.*

In [ ]:
audit("protected_area_status")

In [ ]:
group_overview("Project risk")

# Summary

One row per variable, grouped as above. **Role in CIAT's package** is read from the pinned package (it follows CIAT's updates):
*scores the index (first-pass policy)* for the variables the approved policy uses, *a score in the package, not in the first-pass policy* for the ones CIAT lists as scores but the policy leaves out, *host gate*, *legal mask*, *operational (project risk)*, *evidence only*, or *not in the package*.

In [ ]:
rows = summary_rows(RESULTS, roles, POLICY_VARS)
not_run = [v.id for v in selected if v.id not in RESULTS]
if not_run:
    print("selected but not audited (run their cells above):", not_run)

try:
    import pandas as pd
    table = pd.DataFrame(rows)
    pd.options.display.max_columns = None
    pd.options.display.width = 250
    table.round(3)
except ImportError:
    for r in rows:
        print(f"{r['group']:<18}{r['subgroup']:<26}{r['variable']:<28}{r['origin']:<12}{r['min']:>10.4g}{r['max']:>10.4g}  {r['flags']}")

In [ ]:
import csv

from IPython.display import Markdown

# a table to paste into a document, and the same rows as a csv
head = ["group", "subgroup", "variable", "unit", "dataset", "origin", "min", "max", "mean", "unknown_pct", "role_in_ciat_package", "flags"]
fmt = lambda x: (f"{x:.4g}" if isinstance(x, float) else str(x))
md_lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)]
md_lines += ["| " + " | ".join(fmt(r[h]) for h in head) + " |" for r in rows]
(OUT_DIR / "summary.md").write_text("\n".join(md_lines) + "\n", encoding="utf-8")
with open(OUT_DIR / "summary.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]) if rows else head)
    w.writeheader()
    w.writerows(rows)

flagged = [r for r in rows if r["flags"]]
print(f"{len(rows)} variables audited; {len(flagged)} with something to check:")
for r in flagged:
    print(f"  {r['variable']}: {r['flags']}")
Markdown("\n".join(md_lines))

## Save the audited variables

Writes every audited variable on the grid into one file, so a later step can read them without asking Hugging Face or
Earth Engine again. Run it once the variables above look right.

In [ ]:
import numpy as np

arrays = {vid: a.values.astype("float32") for vid, a in RESULTS.items() if a.values is not None}
np.savez_compressed(OUT_DIR / "variables.npz", **arrays, grid_bbox=np.array(grid.bbox), grid_resolution_deg=grid.resolution_deg)
print("saved", len(arrays), "variables to", (OUT_DIR / "variables.npz").relative_to(REPO_ROOT))